# TUSS / SBOC Crosswalk of Breast Cancer
## Prerequisite for Data Treatment

This notebook builds the **crosswalk** between the 4 SBOC breast cancer
guidelines and the real TUSS reference tables (drugs, procedures, procedure
groups) published by ANS. The output is a list of TUSS codes
(`CD_TABELA_REFERENCIA` + `CD_PROCEDIMENTO`) that the data treatment
notebook (`04_data_treatment.ipynb`) uses to delimit the North region raw base strictly to breast
cancer-related procedures and drugs.

### Why this has to exist as its own notebook, not embedded in treatment

This step requires this categorization to be **validated by
at least two team members before it is used in modeling**, it cannot be a
step that silently re-runs and re-approves itself every time someone
executes the full pipeline. Keeping it as its own notebook, with its own
reviewable output file, makes that validation checkpoint an explicit,
visible action rather
than something baked invisibly into a larger run.

### Inputs

- **TUSS reference tables** (drugs = table 20, procedures = table 22,
  procedure groups = table 63): downloaded directly from the public ANS
  Open Data FTP.
- **The 4 SBOC breast cancer guideline PDFs** (Staging v9, Neoadjuvant v6,
  Adjuvant v6, Advanced Disease v12), stored in
  `documents/extras/sboc_guidelines/` — the only manually-provided input files in
  the whole pipeline, everything else is fetched or produced automatically.
  This notebook does **not** parse the PDFs programmatically, the clinical term list below was extracted by a careful manual reading of the
  4 guidelines, which is the appropriate level of rigor for a clinical
  categorization (an automated PDF-text extraction would be a step
  backward in reliability, not an improvement). The PDFs stay in that
  folder specifically so anyone reviewing or re-validating this list can
  check it against the original source.

### Output

`TUSS_SBOC_Crosswalk_Breast_Cancer.csv`, written to `data/reference/`, one
row per (TUSS table, TUSS code) match, with the SBOC treatment line(s) and
clinical subtype(s).

## Phase 0 — Setup

In [1]:
import io
import re
from pathlib import Path
import unicodedata
import urllib.request
import zipfile

import pandas as pd

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 160)

WORKING_DIR = Path.cwd().resolve()
REPO_ROOT = WORKING_DIR.parent if WORKING_DIR.name == "notebooks" else WORKING_DIR
REFERENCE_DIR = REPO_ROOT / "data" / "reference"
REFERENCE_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_PATH = REFERENCE_DIR / "tuss_sboc_crosswalk_breast_cancer.csv"

## Phase 1 — Downloading the TUSS reference tables

The Unified Terminology for Private Health Insurance (TUSS) is the code
dictionary ANS requires operators and providers to use when billing
procedures, drugs, and medical materials. It is what turns a billed
`CD_PROCEDIMENTO` code in the TISS data into a readable clinical term, and used the other way around, which is what this notebook does, it is what
lets us go from clinical terms cited in the SBOC guidelines to the actual
billing codes to filter by.

We download it fresh from the public ANS FTP (small file, ~1.4 MB) rather
than depending on a copy pre-staged in Drive, so this notebook stays
reproducible on its own.

In [2]:
TUSS_URL = "https://dadosabertos.ans.gov.br/FTP/PDA/terminologia_unificada_saude_suplementar_TUSS-049/pda-049-tuss.zip"

with urllib.request.urlopen(TUSS_URL) as response:
    tuss_zip_bytes = response.read()

with zipfile.ZipFile(io.BytesIO(tuss_zip_bytes)) as z:
    names = z.namelist()
    table_20_name = [n for n in names if "tabela_20_medicamentos" in n][0]
    table_22_name = [n for n in names if "tabela_22_procedimentos" in n][0]
    table_63_name = [n for n in names if "tabela_63_grupos" in n][0]

    with z.open(table_20_name) as f:
        tuss_drugs = pd.read_csv(f, sep=";", encoding="utf-8-sig", dtype=str)
    with z.open(table_22_name) as f:
        tuss_procedures = pd.read_csv(f, sep=";", encoding="utf-8-sig", dtype=str)
    with z.open(table_63_name) as f:
        tuss_groups = pd.read_csv(f, sep=";", encoding="utf-8-sig", dtype=str)

print(f"Table 20 (drugs): {len(tuss_drugs):,} rows")
print(f"Table 22 (procedures): {len(tuss_procedures):,} rows")
print(f"Table 63 (procedure groups): {len(tuss_groups):,} rows")

Table 20 (drugs): 13,451 rows
Table 22 (procedures): 5,966 rows
Table 63 (procedure groups): 31 rows


## Phase 2 — Clinical terms extracted from the SBOC guidelines

The list below was built by reading the 4 SBOC breast cancer guidelines
(Staging v9, Neoadjuvant v6, Adjuvant v6, Advanced Disease v12) (1) and
extracting the drug and procedure names cited for each treatment line
(staging / neoadjuvant / adjuvant / advanced) and, where the guideline
specifies it, for a clinical subtype (HER2+, HR+/HER2-, triple-negative,
BRCA-mutated, NTRK+). Brand names not literally present in the guideline
text are marked `brand_name` in the `source` field, they were added
because the TUSS drug table (table 20) indexes by commercial name, not
generic name.

In [3]:
DRUGS = [
    # (search_term, treatment_line, subtype, source)
    ("doxorrubicina", "neoadjuvant,adjuvant,advanced", "general", "guideline"),
    ("adriamicina", "neoadjuvant,adjuvant", "general", "guideline"),
    ("ciclofosfamida", "neoadjuvant,adjuvant,advanced", "general", "guideline"),
    ("filgrastim", "neoadjuvant,adjuvant", "general", "guideline"),
    ("pegfilgrastim", "neoadjuvant,adjuvant", "general", "guideline"),
    ("paclitaxel", "neoadjuvant,adjuvant,advanced", "general", "guideline"),
    ("docetaxel", "neoadjuvant,adjuvant,advanced", "general", "guideline"),
    ("carboplatina", "neoadjuvant,adjuvant,advanced", "general", "guideline"),
    ("epirrubicina", "neoadjuvant,advanced", "general", "guideline"),
    ("fluorurac", "adjuvant,advanced", "general", "guideline"),
    ("tamoxifeno", "neoadjuvant,adjuvant,advanced", "HR+/HER2-", "guideline"),
    ("letrozol", "neoadjuvant,adjuvant,advanced", "HR+/HER2-", "guideline"),
    ("anastrozol", "neoadjuvant,adjuvant,advanced", "HR+/HER2-", "guideline"),
    ("exemestano", "neoadjuvant,adjuvant,advanced", "HR+/HER2-", "guideline"),
    ("pembrolizumabe", "neoadjuvant,adjuvant,advanced", "triple-negative", "guideline"),
    ("capecitabina", "neoadjuvant,adjuvant,advanced", "general", "guideline"),
    ("olaparibe", "neoadjuvant,adjuvant,advanced", "BRCA-mutated", "guideline"),
    ("trastuzumabe", "staging,neoadjuvant,adjuvant,advanced", "HER2+", "guideline"),
    ("pertuzumabe", "staging,neoadjuvant,adjuvant,advanced", "HER2+", "guideline"),
    ("entansina", "staging,neoadjuvant,adjuvant,advanced", "HER2+", "guideline"),
    ("lapatinibe", "advanced", "HER2+", "guideline"),
    ("metotrexato", "adjuvant,advanced", "general", "guideline"),
    ("neratinibe", "adjuvant", "HER2+", "guideline"),
    ("abemaciclibe", "adjuvant,advanced", "HR+/HER2-", "guideline"),
    ("palbociclibe", "adjuvant,advanced", "HR+/HER2-", "guideline"),
    ("ribociclibe", "adjuvant,advanced", "HR+/HER2-", "guideline"),
    ("triptorrelina", "adjuvant", "HR+/HER2-", "guideline"),
    ("goserelina", "adjuvant", "HR+/HER2-", "guideline"),
    ("zoledronico", "adjuvant", "general", "guideline"),
    ("denosumabe", "adjuvant", "general", "guideline"),
    ("fulvestranto", "advanced", "HR+/HER2-", "guideline"),
    ("megestrol", "advanced", "HR+/HER2-", "guideline"),
    ("everolimo", "advanced", "HR+/HER2-", "guideline"),
    ("alpelisibe", "advanced", "HR+/HER2- PIK3CA-mut", "guideline"),
    ("capivasertibe", "advanced", "HR+/HER2- AKT-mut", "guideline"),
    ("elacestranto", "advanced", "HR+/HER2- (not approved in Brazil)", "guideline"),
    ("deruxtecana", "advanced", "HER2+/HER2-low", "guideline"),
    ("govitecana", "advanced", "triple-negative", "guideline"),
    ("tucatinibe", "advanced", "HER2+ (not approved in Brazil)", "guideline"),
    ("vinorelbina", "advanced", "general", "guideline"),
    ("gencitabina", "advanced", "general", "guideline"),
    ("eribulina", "advanced", "general", "guideline"),
    ("cisplatina", "advanced", "general", "guideline"),
    ("bevacizumabe", "advanced", "general", "guideline"),
    ("atezolizumabe", "advanced", "triple-negative", "guideline"),
    ("larotrectinibe", "advanced", "NTRK+", "guideline"),
    ("talazoparibe", "advanced", "BRCA-mutated", "guideline"),
    # brand names (auxiliary -- NOT literally in the guideline text; see Phase 3)
    ("enhertu", "advanced", "HER2+/HER2-low", "brand_name"),
    ("herceptin", "staging,neoadjuvant,adjuvant,advanced", "HER2+", "brand_name"),
    ("perjeta", "staging,neoadjuvant,adjuvant,advanced", "HER2+", "brand_name"),
    ("kadcyla", "staging,neoadjuvant,adjuvant,advanced", "HER2+", "brand_name"),
    ("trodelvy", "advanced", "triple-negative", "brand_name"),
    ("verzenios", "adjuvant,advanced", "HR+/HER2-", "brand_name"),
    ("kisqali", "adjuvant,advanced", "HR+/HER2-", "brand_name"),
    ("ibrance", "adjuvant,advanced", "HR+/HER2-", "brand_name"),
    ("lynparza", "neoadjuvant,adjuvant,advanced", "BRCA-mutated", "brand_name"),
    ("faslodex", "advanced", "HR+/HER2-", "brand_name"),
    ("genuxal", "neoadjuvant,adjuvant,advanced", "general", "brand_name"),
    ("keytruda", "neoadjuvant,adjuvant,advanced", "triple-negative", "brand_name"),
    ("tecentriq", "advanced", "triple-negative", "brand_name"),
    ("avastin", "advanced", "general", "brand_name"),
    ("prolia", "adjuvant", "general", "brand_name"),
    ("xgeva", "adjuvant", "general", "brand_name"),
    ("zoladex", "adjuvant", "HR+/HER2-", "brand_name"),
    ("tykerb", "neoadjuvant,advanced", "HER2+", "brand_name"),
    ("vitrakvi", "advanced", "NTRK+", "brand_name"),
    ("talzenna", "advanced", "BRCA-mutated", "brand_name"),
    ("navelbine", "advanced", "general", "brand_name"),
]

PROCEDURES = [
    # (search_term, treatment_line, subtype)
    ("mamografia", "staging,neoadjuvant,adjuvant", "general"),
    ("us - mama", "staging,neoadjuvant", "general"),
    ("biopsia percutanea", "staging,neoadjuvant", "general"),
    ("biopsia da mama", "staging,neoadjuvant", "general"),
    ("biopsia de linfonodo", "staging,neoadjuvant,adjuvant", "general"),
    ("linfonodo sentinela", "staging,neoadjuvant,adjuvant", "general"),
    ("puncao aspirativa", "staging", "general"),
    ("imunoistoquim", "staging,neoadjuvant,adjuvant,advanced", "general"),
    ("her2", "staging,neoadjuvant", "HER2 undetermined"),
    ("hibridizacao", "staging,neoadjuvant", "HER2 undetermined"),
    ("cintilografia ossea", "staging,neoadjuvant,advanced", "general"),
    ("pet dedicado oncologico", "staging,neoadjuvant,advanced", "general"),
    ("tc - torax", "staging,neoadjuvant,advanced", "general"),
    ("tc - abdome", "staging,neoadjuvant,advanced", "general"),
    ("rm - mama", "staging,neoadjuvant", "general"),
    ("rm - cranio", "staging,advanced", "general"),
    ("mastectomia", "neoadjuvant,adjuvant", "general"),
    ("tumorectomia", "neoadjuvant,adjuvant", "general"),
    ("quadrantectomia", "neoadjuvant,adjuvant", "general"),
    ("linfadenectomia axilar", "neoadjuvant,adjuvant", "general"),
    ("esvaziamento axilar", "neoadjuvant,adjuvant", "general"),
    ("radioterapia", "neoadjuvant,adjuvant", "general"),
    ("quimioterapia", "neoadjuvant,adjuvant,advanced", "general"),
    ("densitometria ossea", "adjuvant", "general"),
    ("aconselhamento genetico", "staging,neoadjuvant,adjuvant", "general"),
    ("biopsia liquida", "advanced", "general"),
]

print(f"{len(DRUGS)} drug search terms, {len(PROCEDURES)} procedure search terms.")

68 drug search terms, 26 procedure search terms.


## Phase 3 — Matching the terms against the real TUSS tables

Two real quirks of the TUSS drug table shaped how the terms above were
built, worth documenting because they are not obvious from the guidelines
alone:

1. **Table 20 indexes by brand name, not generic name.** `pertuzumabe`
   (generic) does not appear in the table, `PERJETA` (brand) does. That is
   why the brand-name entries above exist: without them, several guideline
   drugs would silently match nothing.
2. **Imaging procedure names use abbreviated prefixes.** It is not
   "ressonancia magnetica da mama", it is `RM - Mama`; not "tomografia
   computadorizada de torax", it is `TC - Torax`. The search terms above
   were adjusted to match the TUSS table's actual naming convention, found
   by inspecting the table directly rather than guessing from the
   guideline's prose.

In [4]:
def normalize(value):
    if pd.isna(value):
        return ""
    value = str(value).lower()
    return unicodedata.normalize("NFKD", value).encode("ascii", "ignore").decode("ascii")


tuss_drugs["_norm"] = (tuss_drugs["NM_TERMO"].fillna("") + " " + tuss_drugs["NM_APRESENTACAO"].fillna("")).apply(normalize)
tuss_procedures["_norm"] = tuss_procedures["NM_TERMO"].fillna("").apply(normalize)
tuss_groups["_norm"] = tuss_groups["DS_TERMO_DETALHADO"].fillna("").apply(normalize)

rows = []

for term, line, subtype, source in DRUGS:
    hits = tuss_drugs[tuss_drugs["_norm"].str.contains(re.escape(normalize(term)), na=False)]
    for _, hit in hits.iterrows():
        rows.append({
            "search_term": term, "category": "drug", "treatment_line": line,
            "subtype": subtype, "term_source": source, "tuss_table": "20",
            "tuss_code": hit["CD_TERMO"], "tuss_name": hit["NM_TERMO"],
            "tuss_presentation": hit.get("NM_APRESENTACAO", ""),
        })

for term, line, subtype in PROCEDURES:
    hits = tuss_procedures[tuss_procedures["_norm"].str.contains(re.escape(normalize(term)), na=False)]
    for _, hit in hits.iterrows():
        rows.append({
            "search_term": term, "category": "procedure", "treatment_line": line,
            "subtype": subtype, "term_source": "guideline", "tuss_table": "22",
            "tuss_code": hit["CD_TERMO"], "tuss_name": hit["NM_TERMO"], "tuss_presentation": "",
        })
    hits_groups = tuss_groups[tuss_groups["_norm"].str.contains(re.escape(normalize(term)), na=False)]
    for _, hit in hits_groups.iterrows():
        rows.append({
            "search_term": term, "category": "procedure_group", "treatment_line": line,
            "subtype": subtype, "term_source": "guideline", "tuss_table": "63",
            "tuss_code": hit["CD_TERMO"], "tuss_name": hit["DS_TERMO_DETALHADO"], "tuss_presentation": "",
        })

crosswalk = pd.DataFrame(rows).drop_duplicates(subset=["tuss_table", "tuss_code", "search_term"])
print(f"Total candidate rows: {len(crosswalk)}")
print(f"Unique (table, code) pairs: {crosswalk[['tuss_table', 'tuss_code']].drop_duplicates().shape[0]}")

Total candidate rows: 401
Unique (table, code) pairs: 400


## Phase 4 — Specificity classification

Not every matched code is equally trustworthy as a breast-cancer signal.
Generic oncology staging exams (chest/abdomen CT, brain MRI, bone
scintigraphy, PET, bone densitometry) are used across dozens of other
cancer types, TISS Ambulatorial has no CID field to confirm they relate
to breast cancer specifically for a given event. We tag these as
`generic_staging` and keep them separate from `specific` codes (drugs and
procedures that only make clinical sense tied to breast cancer, or to
treatment lines explicitly scoped to it by the guidelines).

Two ambiguous procedure terms (`biopsia percutanea`, `biopsia de
linfonodo`) can refer to a biopsy of **any** body site in the raw TUSS
table, not just breast/axilla; we restrict them to only the matches whose
TUSS name explicitly mentions breast or axilla.

**This classification is exactly what feeds the inclusion decision in the
data treatment notebook**: it keeps only `specific` codes, discarding
`generic_staging` entirely (a decision made explicitly with the team,
consistent with the single-state pilot's original approach).

In [5]:
GENERIC_STAGING_TERMS = {
    "tc - torax", "tc - abdome", "rm - cranio", "cintilografia ossea",
    "pet dedicado oncologico", "densitometria ossea",
}
AMBIGUOUS_RESTRICT_TO_BREAST = {"biopsia percutanea", "biopsia de linfonodo"}


def specificity(row):
    return "generic_staging" if row["search_term"] in GENERIC_STAGING_TERMS else "specific"


crosswalk["specificity"] = crosswalk.apply(specificity, axis=1)

is_ambiguous = crosswalk["search_term"].isin(AMBIGUOUS_RESTRICT_TO_BREAST)
mentions_breast = crosswalk["tuss_name"].apply(normalize).str.contains("mama|axil", regex=True, na=False)
crosswalk = crosswalk[~is_ambiguous | (is_ambiguous & mentions_breast)]

crosswalk = crosswalk.sort_values(["specificity", "category", "search_term", "tuss_code"]).reset_index(drop=True)

print(crosswalk["specificity"].value_counts())
print()
print(crosswalk["category"].value_counts())

specificity
specific           385
generic_staging     12
Name: count, dtype: int64

category
drug               322
procedure           73
procedure_group      2
Name: count, dtype: int64


### Search terms with zero matches

Worth checking explicitly: a search term with no TUSS match at all could
mean the drug/procedure genuinely is not billed under that exact name, or
it could mean a naming mismatch was missed. Either way, this should be a
visible check.

In [6]:
terms_found = set(crosswalk["search_term"])
all_terms = {t[0] for t in DRUGS} | {t[0] for t in PROCEDURES}
terms_not_found = sorted(all_terms - terms_found)

if terms_not_found:
    print("Search terms with NO match in the TUSS tables:")
    for t in terms_not_found:
        print(f"  - {t}")
else:
    print("Every search term matched at least one TUSS code.")

Search terms with NO match in the TUSS tables:
  - abemaciclibe
  - adriamicina
  - alpelisibe
  - atezolizumabe
  - bevacizumabe
  - biopsia da mama
  - biopsia de linfonodo
  - biopsia liquida
  - capivasertibe
  - ciclofosfamida
  - denosumabe
  - deruxtecana
  - elacestranto
  - entansina
  - esvaziamento axilar
  - goserelina
  - govitecana
  - lapatinibe
  - larotrectinibe
  - megestrol
  - neratinibe
  - olaparibe
  - pegfilgrastim
  - pembrolizumabe
  - pertuzumabe
  - ribociclibe
  - talazoparibe
  - triptorrelina
  - tucatinibe
  - tumorectomia
  - vinorelbina


In [7]:
# A term with no direct TUSS hit is not necessarily absent from the crosswalk:
# table 20 indexes drugs by brand name (Phase 3), so a generic name can be
# represented through its brand entry. The two cases are separated explicitly
# here, because only the second one is a real coverage gap.
BRAND_ALIASES = {
    "abemaciclibe": ["verzenios"], "adriamicina": ["doxorrubicina"],
    "atezolizumabe": ["tecentriq"], "bevacizumabe": ["avastin"],
    "ciclofosfamida": ["genuxal"], "denosumabe": ["prolia", "xgeva"],
    "deruxtecana": ["enhertu"], "entansina": ["kadcyla"], "goserelina": ["zoladex"],
    "govitecana": ["trodelvy"], "lapatinibe": ["tykerb"], "larotrectinibe": ["vitrakvi"],
    "olaparibe": ["lynparza"], "pembrolizumabe": ["keytruda"], "pertuzumabe": ["perjeta"],
    "ribociclibe": ["kisqali"], "talazoparibe": ["talzenna"], "vinorelbina": ["navelbine"],
}

covered = {t: [b for b in BRAND_ALIASES.get(t, []) if b in terms_found] for t in terms_not_found}
represented = {t: b for t, b in covered.items() if b}
genuinely_absent = [t for t, b in covered.items() if not b]

print(f"Unmatched search terms: {len(terms_not_found)}")
print(f"  represented through a brand-name entry that did match: {len(represented)}")
for term, brands in represented.items():
    print(f"    - {term} -> {', '.join(brands)}")
print(f"  with no representation in the crosswalk at all: {len(genuinely_absent)}")
for term in genuinely_absent:
    print(f"    - {term}")

drugs_absent = [t for t in genuinely_absent if t in {d[0] for d in DRUGS}]
procedures_absent = [t for t in genuinely_absent if t in {p[0] for p in PROCEDURES}]
print(f"\nOf the unrepresented terms, {len(drugs_absent)} are drugs and "
      f"{len(procedures_absent)} are procedures: {procedures_absent}")


Unmatched search terms: 31
  represented through a brand-name entry that did match: 18
    - abemaciclibe -> verzenios
    - adriamicina -> doxorrubicina
    - atezolizumabe -> tecentriq
    - bevacizumabe -> avastin
    - ciclofosfamida -> genuxal
    - denosumabe -> prolia, xgeva
    - deruxtecana -> enhertu
    - entansina -> kadcyla
    - goserelina -> zoladex
    - govitecana -> trodelvy
    - lapatinibe -> tykerb
    - larotrectinibe -> vitrakvi
    - olaparibe -> lynparza
    - pembrolizumabe -> keytruda
    - pertuzumabe -> perjeta
    - ribociclibe -> kisqali
    - talazoparibe -> talzenna
    - vinorelbina -> navelbine
  with no representation in the crosswalk at all: 13
    - alpelisibe
    - biopsia da mama
    - biopsia de linfonodo
    - biopsia liquida
    - capivasertibe
    - elacestranto
    - esvaziamento axilar
    - megestrol
    - neratinibe
    - pegfilgrastim
    - triptorrelina
    - tucatinibe
    - tumorectomia

Of the unrepresented terms, 8 are drugs and 5 a

**Reading of the zero-match terms.** Of the 31 search terms with no direct TUSS
hit, **18 are still represented in the crosswalk through a brand-name entry**
(`pertuzumabe` via `PERJETA`, `olaparibe` via `LYNPARZA`, `ciclofosfamida` via
`GENUXAL`, and so on) — the consequence of table 20 indexing drugs commercially,
already documented in Phase 3. These are not coverage gaps.

**13 terms have no representation at all.** Five are procedures
(`biopsia da mama`, `biopsia de linfonodo`, `biopsia liquida`, `esvaziamento
axilar`, `tumorectomia`) whose clinical content is largely reached by the
sibling terms that did match (`biopsia percutanea`, `linfonodo sentinela`,
`mastectomia`, `quadrantectomia`, `linfadenectomia axilar`), so the scope loss
is small but real. The remaining eight are drugs: `elacestranto` and
`tucatinibe` are flagged in Phase 2 as not approved in Brazil and are expected
to be absent; `alpelisibe`, `capivasertibe`, `megestrol`, `neratinibe`,
`pegfilgrastim` and `triptorrelina` are approved agents that simply do not
appear in the TUSS tables under any name tested here.

**What this means for the filter.** The absent drugs are low-volume, later-line
agents, so their omission narrows the scope slightly rather than biasing it
toward a treatment line; the absent procedures overlap with terms that did
match. The list is printed rather than summarised so a reviewer can re-check it
against the guidelines, and it is the first place to extend if the crosswalk is
revised.

## Phase 5 — Export

Written to `data/reference/`, so the data treatment notebook
(`04_data_treatment.ipynb`) can read it directly
without any manual path configuration.

In [8]:
crosswalk.to_csv(OUTPUT_PATH, index=False, encoding="utf-8-sig")
print(f"Saved to: {OUTPUT_PATH}")
print(f"Total rows: {len(crosswalk)}")
crosswalk.head()

Saved to: C:\Users\Inteli\Documents\Projetos\1Ano_Módulo3\g02\data\reference\tuss_sboc_crosswalk_breast_cancer.csv
Total rows: 397


,search_term,category,treatment_line,subtype,term_source,tuss_table,tuss_code,tuss_name,tuss_presentation,specificity
0,cintilografia ossea,procedure,"staging,neoadjuvant,advanced",general,guideline,22,40706010,Cintilografia óssea (corpo total),,generic_staging
1,densitometria ossea,procedure,adjuvant,general,guideline,22,40808122,Densitometria óssea (um segmento),,generic_staging
2,densitometria ossea,procedure,adjuvant,general,guideline,22,40808130,Densitometria óssea - rotina: coluna e fêmur (...,,generic_staging
3,densitometria ossea,procedure,adjuvant,general,guideline,22,40808149,Densitometria óssea - corpo inteiro (avaliação...,,generic_staging
4,densitometria ossea,procedure,adjuvant,general,guideline,22,40808343,Revisão de densitometria óssea por procedimento,,generic_staging


## Summary

| Item | Result |
|---|---|
| SBOC guidelines used | Staging (v9), Neoadjuvant (v6), Adjuvant (v6), Advanced Disease (v12) |
| Drug search terms | see Phase 2 |
| Procedure search terms | see Phase 2 |
| TUSS tables searched | 20 (drugs), 22 (procedures), 63 (procedure groups) |
| Specificity tags | `specific` (kept for the inclusion filter of notebook 04) vs. `generic_staging` (documented, not used for inclusion) |
| Output | `data/reference/tuss_sboc_crosswalk_breast_cancer.csv` |

### References

1. Sociedade Brasileira de Oncologia Clinica (SBOC). SBOC 2025 Guidelines - Breast Cancer: Staging (v9), Neoadjuvant (v6), Adjuvant (v6), Advanced Disease (v12). Sao Paulo: SBOC; 2025.
2. Agencia Nacional de Saude Suplementar (ANS). Terminologia Unificada da Saude Suplementar (TUSS) [Internet]. Brasilia: ANS; 2026 [cited 2026 Aug 14]. Available from: https://dadosabertos.ans.gov.br/FTP/PDA/terminologia_unificada_saude_suplementar_TUSS-049/